<a href="https://colab.research.google.com/github/acerNZ/HAL/blob/master/ReqGap_Analysis_Col_mapping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ========================================================
#  ULTIMATE: FULLY CONFIGURABLE TRACEABILITY
#  FIXED: No ValueError on SelectMultiple
#  Google Colab — Upload → Map → Run
# ========================================================

# --- INSTALL ---
import subprocess, sys
def install(p): subprocess.check_call([sys.executable, "-m", "pip", "install", p], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print("Installing packages...")
install("pandas"); install("openpyxl"); install("fuzzywuzzy"); install("python-levenshtein"); install("nltk")

import pandas as pd, io, base64, ipywidgets as widgets, re, nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from fuzzywuzzy import fuzz
from IPython.display import display, HTML, clear_output

# --- TEXT CLEANING ---
stop_words = set(stopwords.words('english'))
def clean_text(text):
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', str(text).lower())
    try:
        tokens = word_tokenize(text)
    except:
        tokens = text.split()
    return [t for t in tokens if t not in stop_words and len(t) > 2]

# --- COMPARISON ---
def compare_english_vs_gherkin(req_text, gherkin):
    req_clean = clean_text(req_text)
    gherkin_clean = clean_text(gherkin)
    overlap = len(set(req_clean) & set(gherkin_clean))
    jaccard = overlap / len(set(req_clean) | set(gherkin_clean)) if req_clean or gherkin_clean else 0
    fuzzy_score = fuzz.partial_ratio(str(req_text).lower(), str(gherkin).lower()) / 100
    keywords = ['must', 'shall', 'button', 'lookup', 'mandatory', 'system generated', 'auto-populated', 'save', 'edit', 'display']
    kw_boost = sum(1 for kw in keywords if kw in str(req_text).lower() and kw in str(gherkin).lower()) * 0.05
    final_score = int((jaccard * 60) + (fuzzy_score * 30) + (kw_boost * 100))
    final_score = min(final_score, 100)
    color = "green" if final_score >= 90 else "orange" if final_score >= 70 else "red"
    return final_score, f"<span style='color:{color}'>Score: {final_score}</span>", overlap

# --- EXTRACT ALL TEXT ---
def extract_all_text(row, cols):
    return " | ".join([str(row.get(c, "")).strip() for c in cols if pd.notna(row.get(c)) and str(row.get(c)).strip()])

# --- MATCHING LOGIC ---
def match_and_compare(row, stories_df, req_id_col, req_text_cols, req_desc_cols, req_tag_cols, story_id_col, story_ac_cols, story_desc_cols, manual_col):
    req_id = str(row.get(req_id_col, "")) if req_id_col else ""
    req_text = extract_all_text(row, req_text_cols)
    req_desc = extract_all_text(row, req_desc_cols)
    req_tags = extract_all_text(row, req_tag_cols)
    req_all = f"{req_text} {req_desc} {req_tags}".strip()

    manual = str(row.get(manual_col, '')).strip() if manual_col else ""
    if manual and manual not in ['nan', '', 'None']:
        ids = [x.strip() for x in re.split(r'[/,;\s]+', manual) if x.strip() and len(x.strip()) >= 4]
        valid = [id for id in ids if id in stories_df['detected_id'].astype(str).values]
        if valid:
            results = []
            for vid in valid:
                story = stories_df[stories_df['detected_id'] == vid].iloc[0]
                story_text = extract_all_text(story, story_ac_cols + story_desc_cols)
                score, _, _ = compare_english_vs_gherkin(req_all, story_text)
                results.append(f"{vid} (Score: {score})")
            return ", ".join(results), "MANUAL", story_text[:200]

    # Auto-match
    req_lower = req_all.lower()
    req_ids = re.findall(r'\b([a-z]{2}\.[a-z]{2}\.[a-z]{3}\.\d{4})\b', req_lower) + re.findall(r'\b(\d{5,})\b', req_lower)
    req_ids = list(set(req_ids))

    best_match = None
    best_score = 0
    best_gherkin = ""

    for _, story in stories_df.iterrows():
        story_id = story['detected_id']
        story_text = extract_all_text(story, story_ac_cols + story_desc_cols)
        story_lower = story_text.lower()

        if any(rid in story_lower for rid in req_ids):
            score, _, _ = compare_english_vs_gherkin(req_all, story_text)
            return f"{story_id} (Score: {score})", "AUTO-ID", story_text[:200]

        overlap_words = len(set(clean_text(req_all)) & set(clean_text(story_text)))
        hier_match = any(part in story_lower for part in str(row['Hierarchy']).lower().split("→") if len(part) > 3)
        tag_match = any(t in story_lower for t in [x.strip() for x in req_tags.split("|") if len(x.strip()) > 2])

        score = (hier_match * 30) + (tag_match * 40) + (overlap_words * 5)
        if score > best_score:
            best_score = score
            best_match = story_id
            best_gherkin = story_text[:200]

    if best_score >= 50:
        score, _, _ = compare_english_vs_gherkin(req_all, stories_df[stories_df['detected_id'] == best_match].iloc[0][story_ac_cols[0]])
        return f"{best_match} (Score: {score})", "AUTO", best_gherkin

    return "GAP", "GAP", ""

# --- UI ---
story_upload = widgets.FileUpload(accept='.xlsx,.csv,.xls', description="User Stories")
req_upload = widgets.FileUpload(accept='.xlsx,.csv,.xls', description="Requirements")
status = widgets.Output(); output = widgets.Output()

# Dropdowns
story_id = widgets.Dropdown(options=[], description='Story ID:')
story_ac = widgets.SelectMultiple(options=[], description='AC (Gherkin):', rows=4)
story_desc = widgets.SelectMultiple(options=[], description='Description/Notes:', rows=4)
req_id = widgets.Dropdown(options=[], description='Req ID:')
req_text = widgets.SelectMultiple(options=[], description='Req Text:', rows=4)
req_desc = widgets.SelectMultiple(options=[], description='Req Desc/Notes:', rows=4)
req_tags = widgets.SelectMultiple(options=[], description='Tags/Refs:', rows=4)
req_manual = widgets.Dropdown(options=['(None)'], description='Manual Link:')
req_title = widgets.SelectMultiple(options=[], description='Hierarchy (Title 1..N):', rows=5)
run_button = widgets.Button(description="RUN TRACEABILITY", button_style='success')

stories_df = None
reqs_df = None

def load_stories(change):
    global stories_df
    if story_upload.value:
        s_info = list(story_upload.value.values())[0]
        s_io = io.BytesIO(s_info['content'])
        stories_df = pd.read_excel(s_io) if s_info['metadata']['name'].lower().endswith(('.xlsx','.xls')) else pd.read_csv(s_io)
        cols = list(stories_df.columns)
        for w in [story_id, story_ac, story_desc]: w.options = cols
        with status: clear_output(); print("User Stories loaded.")

def load_reqs(change):
    global reqs_df
    if req_upload.value:
        r_info = list(req_upload.value.values())[0]
        r_io = io.BytesIO(r_info['content'])
        reqs_df = pd.read_excel(r_io) if r_info['metadata']['name'].lower().endswith(('.xlsx','.xls')) else pd.read_csv(r_io)
        cols = list(reqs_df.columns)
        for w in [req_id, req_text, req_desc, req_tags, req_title]: w.options = cols
        req_manual.options = ['(None)'] + cols
        with status: clear_output(); print("Requirements loaded.")

def run_analysis(b):
    with output: clear_output(); print("Running...")

    # FIXED: Check SelectMultiple properly
    if (stories_df is None or
        reqs_df is None or
        not story_id.value or
        not story_ac.value or
        not req_text.value):
        print("ERROR: Please select Story ID, at least one AC, and at least one Req Text.")
        return

    try:
        # Extract
        stories_df['detected_id'] = stories_df[story_id.value].astype(str)
        reqs_df['Hierarchy'] = reqs_df.apply(
            lambda row: " → ".join([str(row[c]).strip() for c in req_title.value
                                  if pd.notna(row.get(c)) and str(row.get(c)).strip()]),
            axis=1
        )

        results = []
        for _, row in reqs_df.iterrows():
            covered, method, gherkin = match_and_compare(
                row, stories_df,
                req_id.value, list(req_text.value), list(req_desc.value), list(req_tags.value),
                story_id.value, list(story_ac.value), list(story_desc.value),
                req_manual.value if req_manual.value != '(None)' else None
            )
            results.append({
                'Req_ID': row.get(req_id.value, ''),
                'Req_Text': extract_all_text(row, req_text.value),
                'Hierarchy': row['Hierarchy'],
                'Covered_By': covered,
                'Match_Method': method,
                'Gherkin_Snippet': gherkin,
                'Status': "COVERED" if covered != "GAP" else "GAP"
            })
        result_df = pd.DataFrame(results)

        gaps = result_df[result_df['Status'] == 'GAP']
        print(f"FINAL REPORT")
        print(f"Total: {len(result_df)} | Covered: {len(result_df)-len(gaps)} | GAPS: {len(gaps)}")
        if len(gaps) > 0:
            print(f"\nREAL GAPS:")
            display(gaps[['Req_ID', 'Req_Text']].head(20))

        buf = io.BytesIO()
        result_df.to_excel(buf, index=False, engine='openpyxl')
        buf.seek(0)
        b64 = base64.b64encode(buf.read()).decode()
        link = f'<a href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{b64}" download="FULL_TRACEABILITY.xlsx">DOWNLOAD FULL REPORT</a>'
        display(HTML(f"<br><strong>{link}</strong>"))

    except Exception as e:
        print(f"ERROR: {e}")
        import traceback; traceback.print_exc()

# --- OBSERVE ---
story_upload.observe(load_stories, 'value')
req_upload.observe(load_reqs, 'value')
run_button.on_click(run_analysis)

# --- UI ---
display(HTML("<h3>1. Upload Files</h3>"))
display(story_upload, req_upload)
display(HTML("<h3>2. Map Columns</h3>"))
display(HTML("<b>User Stories:</b>"), widgets.HBox([story_id, story_ac, story_desc]))
display(HTML("<b>Requirements:</b>"), widgets.HBox([req_id, req_text, req_desc]))
display(HTML("<b>Tags & Links:</b>"), widgets.HBox([req_tags, req_manual]))
display(HTML("<b>Hierarchy:</b>"), req_title)
display(run_button)
display(HTML("<b>Status:</b>"), status)
display(HTML("<b>Output:</b>"), output)